In [1]:
import requests
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, pipeline
from pathlib import Path
import glob
import os
from transformers import CLIPProcessor, CLIPModel

In [2]:
!pip install -qU bitsandbytes
import bitsandbytes as bnb
print(bnb.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 34.1 MB/s eta 0:00:00
0.49.2


In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

dino_id = "IDEA-Research/grounding-dino-tiny"
dino_processor = AutoProcessor.from_pretrained(dino_id)
dino = AutoModelForZeroShotObjectDetection.from_pretrained(dino_id).to(device)

preprocessor_config.json:   0%|          | 0.00/457 [00:00<?, ?B/s]

The image processor of type `GroundingDinoImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/1.64k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.24k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/689M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/990 [00:00<?, ?it/s]

In [4]:
clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

In [5]:
# Carga de datos
# Esta celda descarga todos los datos, y los extrae
import gdown, zipfile

url = 'https://drive.google.com/drive/folders/1MvRYh3SudRabydH2eAAzRq2K5Os1j41-'
output_folder = 'recsys-pf'

gdown.download_folder(url, output=output_folder, quiet=False, use_cookies=False)
zip_search_pattern = os.path.join(output_folder, '*.zip')
zip_files = glob.glob(zip_search_pattern)
image_zip_path = zip_files[0]

with zipfile.ZipFile(image_zip_path, 'r') as zip_ref:
    zip_ref.extractall(output_folder)

Retrieving folder contents


Processing file 1_CZsjxiXc-LsiXjRpiuEsGCZdvq7i0au interaction.csv
Processing file 1R95OpzeRwkG0Ljc9MJqKJI3zMSwqItHV item_info.csv
Processing file 1P4kKxL5xgJ_ePj955gIEej78bsnk36M5 resized_images.zip


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1_CZsjxiXc-LsiXjRpiuEsGCZdvq7i0au
To: /content/recsys-pf/interaction.csv
100%|██████████| 28.1M/28.1M [00:00<00:00, 41.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1R95OpzeRwkG0Ljc9MJqKJI3zMSwqItHV
To: /content/recsys-pf/item_info.csv
100%|██████████| 25.0M/25.0M [00:00<00:00, 246MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1P4kKxL5xgJ_ePj955gIEej78bsnk36M5
From (redirected): https://drive.google.com/uc?id=1P4kKxL5xgJ_ePj955gIEej78bsnk36M5&confirm=t&uuid=f7198a58-8852-45b0-b614-da1fd664f483
To: /content/recsys-pf/resized_images.zip
100%|██████████| 163M/163M [00:00<00:00, 265MB/s]
Download completed


In [6]:
import pandas as pd
import numpy as np

interaction_df = pd.read_csv('recsys-pf/interaction.csv')
item_info_df = pd.read_csv('recsys-pf/item_info.csv')

In [7]:
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# LLM (QWEN)
print("Cargando el modelo Qwen2.5-1.5B")
generador_qwen = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    model_kwargs={
        "quantization_config": BitsAndBytesConfig(load_in_4bit=True),
        "device_map": "auto"
        },
    tokenizer=AutoTokenizer.from_pretrained(
      "Qwen/Qwen2.5-1.5B-Instruct", trust_remote_code=True),
    torch_dtype=torch.float16
)

# Definimos los Prompts del Sistema que corregimos
PROMPT_SUMMARIZER = (
    "You are an expert data annotator for computer vision. Your task is to read a noisy "
    "social media post and translate it into a concise, less than 20-word declarative "
    "description of the visual scene it implies.\n"
    "Instructions:\n"
    "- Limit the description to strictly less than 20 words.\n"
    "- Concentrate on capturing visually observable attributes (subjects, setting, clothing, objects). "
    "If they are not explicit, logically infer them from the provided text, emojis, and hashtags.\n"
    "- Refrain from using engaging, subjective, or persuasive language. Use strictly declarative and objective language."
)

PROMPT_EVALUATOR = (
    "You are a summary evaluator for data annotation in computer vision. Your task is to evaluate a "
    "social media post summary according to the following criteria:\n"
    "- The summary must be strictly less than 20 words.\n"
    "- It must encapsulate the visual elements explicitly present in the original description, or "
    "correctly infer visual elements that would logically appear in an accompanying image.\n"
    "- It must use strictly declarative and objective language.\n"
    "- You must provide feedback and revision suggestions focused solely on the presence or absence of visual elements.\n"
    "Instructions:\n"
    "- If you find the summary is too long, ask for a shorter summary.\n"
    "- If you determine that no further revisions are needed and all criteria are met, end your output "
    "with <STOP> (without any extra text)."
)

PROMPT_REFINER = (
    "You are an expert data annotator for computer vision. Your task is to refine a summary based on "
    "the provided feedback. Follow all the suggestions exactly and do not provide any additional commentary. "
    "Give only one final summary as your output.\n"
    "Instructions:\n"
    "- Limit the summary to strictly fewer than 20 words.\n"
    "- Include in your description only elements which are visually observable or logically inferable from the original description.\n"
    "- Use strictly declarative and objective language."
)

def consultar_llm(prompt_sistema, prompt_usuario, max_tokens):
    mensajes = [
        {"role": "system", "content": prompt_sistema},
        {"role": "user", "content": prompt_usuario}
    ]
    salida = generador_qwen(
        mensajes,
        max_new_tokens=max_tokens,
        temperature=0.3,
        do_sample=True
    )
    return salida[0]["generated_text"][-1]["content"].strip()

Cargando el modelo Qwen2.5-1.5B


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [8]:
import glob
import pandas as pd

clip = clip.to(device)
rutas = glob.glob("recsys-pf/resized_images/*.jpg")
rutas_sample = pd.Series(rutas).sample(n=100, random_state=5).tolist()

max_iteraciones = 5 # para el llm
images_features = {}
textual_features = {}

for ruta in rutas_sample:
    item_id = os.path.basename(ruta).replace(".jpg", "")
    image = Image.open(ruta).convert("RGB")
    # Sacar tag asociada a imagen para buscarla con dion
    item_info = item_info_df[ item_info_df["item_id"] == item_id ]
    tags = item_info["tag"].iloc[0].split("·")
    text_labels = [ [ t for  t in tags] ]

    """ Quizá esto no sea tan sencillo,
    estas categorías pueden ser más abstractas que vestido o mochila.
    Cosas como comedia """

    inputs = dino_processor(images=image, text=text_labels, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = dino(**inputs)

    result_boxes = dino_processor.post_process_grounded_object_detection(
        outputs,
        inputs.input_ids,
        threshold=0.4,
        text_threshold=0.3,
        target_sizes=[image.size[::-1]]
    )

    print("\nImagen: ", item_id)
    print("Categoría buscada: ", text_labels[0])
    result_box = result_boxes[0]
    """ for box, score, labels in zip(result["boxes"], result["scores"], result["labels"]):
        box = [round(x, 2) for x in box.tolist()]
        print(f"Detected {labels} with confidence {round(score.item(), 3)} at location {box}") """

    # Recortar imagen a caja encontrada si se puede
    score, box = result_box["scores"], result_box["boxes"]
    if score.numel() != 0:
        x_min, y_min, x_max, y_max = [int(coord) for coord in box[0].cpu().tolist()]
        image = image.crop((x_min, y_min, x_max, y_max))
    else:
        print("no se encontro objeto")

    # obtener features de la imagen con CLIP (explicit to guarantee tensor)

    pixel_values = clip_processor(images=image, return_tensors="pt").to(device).pixel_values
    with torch.no_grad():
        vision_outputs = clip.vision_model(pixel_values=pixel_values)
        image_features = clip.visual_projection(vision_outputs.pooler_output)
    image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
    images_features[item_id] = image_features.cpu().numpy()

    # Creación query con LLM
    textos = tags + [item_info["title"].iloc[0]] + [item_info["description"].iloc[0]]
    texto = " ".join( x for x in textos if isinstance(x, str) )
    prompt_usu_sum = f"Original post: {texto}"
    resumen_actual = consultar_llm(PROMPT_SUMMARIZER, prompt_usu_sum, max_tokens=25)
    prompt_usu_eval = f"Original post: {texto}\nCurrent summary: {resumen_actual}"

    for i in range(max_iteraciones):
        feedback = consultar_llm(PROMPT_EVALUATOR, prompt_usu_eval, max_tokens=50)
        if "<STOP>" in feedback:
                break
        prompt_usu_ref = (
            f"Original post: {texto}\n"
            f"Current summary: {resumen_actual}\n"
            f"Evaluator Feedback: {feedback}"
        )
        resumen_actual = consultar_llm(PROMPT_REFINER, prompt_usu_ref, max_tokens=25)

    print(texto)
    print(resumen_actual, "\n")

    # Obtener features de texto en base a características con CLIP (explicit)
    text_inputs = clip_processor(text=resumen_actual, return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        text_outputs = clip.text_model(**text_inputs)
        text_features = clip.text_projection(text_outputs.pooler_output)
    text_features = text_features / text_features.norm(p=2, dim=-1, keepdim=True)
    textual_features[item_id] = text_features.cpu().numpy()

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=25) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Imagen:  i52788
Categoría buscada:  ['Dogs']


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=25) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

Dogs How the German Shepherd reacts when he realizes that his sheep is missing. Out to walk to see what reaction after Beta found Alpha disappeared did not expect Beta actually so chicken thief!
German Shepherd reacts excitedly upon realizing his sheep is missing; Alpha unexpectedly becomes a chicken thief. 


Imagen:  i300330
Categoría buscada:  ['Short Film', 'Hand-drawn', 'Dubbing']


Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=25) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=25) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation f

...

Variety Shows [Creation 】 Yang beyond the theme song re-rating and straight shot comparison, can be said to be very touching! Compare Yang Beyond's theme song re-rating with a straight shot, it can be said to be a great improvement, touching
Yang Beyond's theme song re-rating surpasses expectations, significantly outperforming a direct comparison. 


Imagen:  i311987
Categoría buscada:  ['Daily Life']


Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=25) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Daily Life [Divine restoration!] School Games slugger New Treasure Island, full of internal flavor, burning up the whole place! The front action is taken from the fierce version of the new treasure island, the back action is taken from the world order, the participants are all members of the senior high school class of Fu'an No. 1 ps: the action is to see the original video bit by bit keyed out, have not seen any instructional video!
A school game session in New Treasure Island, with intense action and diverse characters, captured in real-time footage. 


Imagen:  i309647
Categoría buscada:  ['Rhythm Games']


Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Rhythm Games [My World] VereinCraft Mega War Server Promo Video "This is a promo for the VereinCraft war server, this server was built with years, has a huge tech team, and over 10,000 players [who have played it]
Promotional video showcasing the mega war server by VereinCraft, boasting a large tech team and over 10,0 



In [9]:
# Faltaría el codigo que hace la recomendación
interaction_df

,item_id,user_id,timestamp
0,i72138,u209296,1605059546
1,i15530,u2444520,1628914341
2,i95199,u1866870,1601008921
3,i3413,u2498546,1505731122
4,i224963,u3676118,1643894394
...,...,...,...
989489,i53095,u5426144,1648142361
989490,i219825,u2841982,1605073212
989491,i42184,u2257424,1644932293
989492,i84962,u1879882,1639377159


In [16]:
# Fuse image + text features per item
import numpy as np
fused_features = {}

for item_id in images_features.keys():
    img_feat = images_features[item_id].flatten()
    txt_feat = textual_features[item_id].flatten()

    # Concatenation (different embedding spaces)
    fused = np.concatenate([img_feat, txt_feat])

    fused = fused / np.linalg.norm(fused)
    fused_features[item_id] = fused

print(f"Fused {len(fused_features)} item embeddings")
print("Fused dimension:", fused_features[list(fused_features.keys())[0]].shape)

Fused 100 item embeddings
Fused dimension: (1024,)


In [17]:
# Create user profiles (mean of interacted item fused embeddings)
import numpy as np
import pandas as pd

user_profiles = {}

for user_id, group in interaction_df.groupby('user_id'):
    user_items = group['item_id'].tolist()
    valid_items = [iid for iid in user_items if iid in fused_features]
    if len(valid_items) == 0:
        continue
    # Mean embedding
    user_vec = np.mean([fused_features[iid] for iid in valid_items], axis=0)
    # Normalize
    user_vec = user_vec / np.linalg.norm(user_vec)
    user_profiles[user_id] = user_vec

print(f"Created profiles for {len(user_profiles)} users")

Created profiles for 997 users


In [18]:
# Rank unseen items by cosine similarity to user profile
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def recommend_for_user(user_id, top_n=10):
    if user_id not in user_profiles:
        print(f"No profile for user {user_id}")
        return []

    user_vec = user_profiles[user_id].reshape(1, -1)

    # Get items the user has already interacted with
    interacted = set(interaction_df[interaction_df['user_id'] == user_id]['item_id'].tolist())

    # Candidate items (not interacted)
    candidates = [iid for iid in fused_features.keys() if iid not in interacted]
    if not candidates:
        print("No candidates left")
        return []

    # Compute cosine similarities
    candidate_vecs = np.array([fused_features[iid] for iid in candidates])
    sims = cosine_similarity(user_vec, candidate_vecs)[0]

    # Rank
    ranked_idx = np.argsort(sims)[::-1][:top_n]
    recommendations = [(candidates[i], float(sims[i])) for i in ranked_idx]
    return recommendations

# Example usage - pick a user with profile
sample_user = list(user_profiles.keys())[0]
recs = recommend_for_user(sample_user, top_n=10)
print(f"\nTop-10 recommendations for user {sample_user}:")

for item_id, score in recs:
    # Try to get title if available
    title = item_info_df[item_info_df['item_id'] == item_id]['title'].iloc[0] if len(item_info_df[item_info_df['item_id'] == item_id]) > 0 else item_id
    print(f"  {item_id} | {title[:60] if isinstance(title, str) else item_id} | score={score:.4f}")


Top-10 recommendations for user u10042902:
  i62753 | She was jealously bullied by her female classmates to the po | score=0.8124
  i45688 | The first time I've ever seen a TV show where everyone is be | score=0.7939
  i193367 | Minglan dislikes Red Wolf after disliking Molan for the seco | score=0.7880
  i137890 | The soul of the P picture] how you wear the clothes of the s | score=0.7868
  i256455 | [Lee Jun Ki Evil Flower] How a senseless male god becomes an | score=0.7854
  i133829 | I went to bed at 1:30 a.m. every day of my senior year of hi | score=0.7810
  i234397 | Why is it so popular with foreigners? Your mom will take you | score=0.7795
  i150527 | "[Cai Shaofen's textbook acting]" "Tomie on Earth" "Revenge  | score=0.7775
  i322565 | Calendar Girl is full of energy with you! //Calendar Girls'  | score=0.7728
  i54847 | Savage, off-key. | score=0.7704
